# Qwen3.5-4B Telecom RCA: SFT + Held-Out Evaluation

This Colab notebook fine-tunes `unsloth/Qwen3.5-4B` with 16-bit LoRA on the synthetic telecom RCA reasoning trajectories, evaluates held-out loss during training, and measures generated-label accuracy on all 864 official validation questions.

Key rules:

- The synthetic reasoning is trained in Qwen3.5's native `<think>...</think>` format.
- The validation set contains only known final labels. No validation reasoning is invented.
- Validation examples are never optimizer inputs.
- Qwen3.5 4-bit QLoRA is intentionally disabled because Unsloth currently recommends 16-bit LoRA for this model family.
- GRPO is a separate stage and must reuse this exact base model, tokenizer/chat template, and response contract.


## 1. Install the Qwen3.5-compatible Unsloth stack

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**. The first run compiles Qwen3.5's hybrid-model kernels and can take several minutes.


In [ ]:
%%capture
import importlib.util
import os

!pip install --upgrade -qqq uv

# This follows Unsloth's current free-Colab Qwen3.5 setup. Qwen3.5 requires
# Transformers v5; the T4 path uses 16-bit LoRA rather than 4-bit QLoRA.
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ):
    try:
        import numpy
        import PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pillow = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"
        _pillow = "pillow"
    !uv pip install -qqq --upgrade \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pillow} torchvision \
        bitsandbytes "xformers==0.0.32.post2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install -qqq --upgrade --no-deps \
    "tokenizers>=0.22.0,<=0.23.0" "trl==0.22.2" unsloth unsloth_zoo
!uv pip install -qqq "transformers==5.2.0"
!uv pip install -qqq --no-build-isolation flash-linear-attention "causal_conv1d==1.6.0"
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"


## 2. Configuration and input files

Upload these two files into the Colab working directory:

- `sft_train_data.jsonl` — the generated synthetic `question`/`response` data.
- `sft_validation_data.jsonl` — the prepared 864-row held-out validation artifact.

The notebook also checks `/content/drive/MyDrive/` if the files are not present locally.


In [ ]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
OUTPUT_DIR = "qwen35_4b_sft_outputs"
ADAPTER_DIR = "qwen35_4b_sft_lora"
GENERATION_MAX_NEW_TOKENS = 1024

# Set to a positive integer for a quick smoke test. Keep None for the official
# all-864-question evaluation.
EVAL_LIMIT = None

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


def locate_file(filename):
    candidates = [
        Path(filename),
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to Colab or place it in MyDrive."
    )


TRAIN_PATH = locate_file("sft_train_data.jsonl")
VALIDATION_PATH = locate_file("sft_validation_data.jsonl")
print("Training data:", TRAIN_PATH)
print("Validation data:", VALIDATION_PATH)


## 3. Load Qwen3.5-4B and attach language-only LoRA

`fast_inference=False` is deliberate: current Qwen3.5 reinforcement learning and training should use Unsloth inference rather than the older vLLM path. Vision parameters remain frozen because this task is text-only.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loaded:", MODEL_NAME)


## 4. Validate and format the datasets

Training responses are converted from their existing four-section form into:

```text
<think>
...synthetic reasoning...
</think>

\boxed{R#}
```

Validation responses remain answer-only (`\boxed{C#}`). They support completion-only held-out loss without fabricating reasoning that is absent from the source data.


In [ ]:
TRAIN_BOX_RE = re.compile(r"\\boxed\{(R[1-8])\}\s*$")
VALID_RESPONSE_RE = re.compile(r"\\boxed\{(C[1-8])\}")


def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number} is empty")
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error
            if not isinstance(item, dict):
                raise ValueError(f"{path}:{line_number} must be a JSON object")
            records.append(item)
    return records


def format_training_response(response, location):
    if not isinstance(response, str) or not response.strip():
        raise ValueError(f"{location}: response must be non-empty")
    matches = list(TRAIN_BOX_RE.finditer(response))
    all_boxes = re.findall(r"\\boxed\{R[1-8]\}", response)
    if len(matches) != 1 or len(all_boxes) != 1:
        raise ValueError(f"{location}: expected exactly one terminal boxed R1-R8 label")
    match = matches[0]
    reasoning = response[: match.start()].rstrip()
    if not reasoning:
        raise ValueError(f"{location}: reasoning is empty")
    return f"<think>\n{reasoning}\n</think>\n\n\\boxed{{{match.group(1)}}}"


def conversation(question, assistant_response):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_response},
    ]


raw_train = load_jsonl(TRAIN_PATH)
raw_validation = load_jsonl(VALIDATION_PATH)
assert len(raw_train) == 1941, f"Expected 1,941 SFT examples; got {len(raw_train)}"
assert len(raw_validation) == 864, f"Expected 864 validation examples; got {len(raw_validation)}"

train_rows = []
train_questions = set()
for index, item in enumerate(raw_train):
    if set(item) != {"question", "response"}:
        raise ValueError(f"train row {index}: expected only question and response")
    question = item["question"]
    if not isinstance(question, str) or not question.strip() or question in train_questions:
        raise ValueError(f"train row {index}: empty or duplicate question")
    response = format_training_response(item["response"], f"train row {index}")
    train_rows.append({"messages": conversation(question, response)})
    train_questions.add(question)

validation_rows = []
validation_questions = set()
validation_labels = Counter()
for index, item in enumerate(raw_validation):
    if set(item) != {"id", "question", "response"}:
        raise ValueError(f"validation row {index}: expected id, question, and response")
    identifier = item["id"]
    question = item["question"]
    response = item["response"]
    match = VALID_RESPONSE_RE.fullmatch(response)
    if not identifier or not question or not match:
        raise ValueError(f"validation row {index}: malformed id, question, or response")
    if question in validation_questions or question in train_questions:
        raise ValueError(f"validation row {index}: duplicate or train/validation overlap")
    label = match.group(1)
    validation_rows.append(
        {
            "id": identifier,
            "question": question,
            "target": label,
            "messages": conversation(question, response),
        }
    )
    validation_questions.add(question)
    validation_labels[label] += 1

expected_balance = Counter({f"C{i}": 108 for i in range(1, 9)})
assert validation_labels == expected_balance, validation_labels
print(f"Validated {len(train_rows):,} SFT and {len(validation_rows):,} held-out examples")
print("Validation balance:", dict(sorted(validation_labels.items())))
print("Example formatted response tail:")
print(train_rows[0]["messages"][-1]["content"][-400:])


In [ ]:
def render_conversation(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


train_text_rows = [{"text": render_conversation(row["messages"])} for row in train_rows]
validation_text_rows = [
    {"text": render_conversation(row["messages"])} for row in validation_rows
]
train_dataset = Dataset.from_list(train_text_rows)
validation_dataset = Dataset.from_list(validation_text_rows)


def token_length(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


train_lengths = [token_length(row["text"]) for row in train_text_rows]
validation_lengths = [token_length(row["text"]) for row in validation_text_rows]
max_observed = max(train_lengths + validation_lengths)
if max_observed > MAX_SEQ_LENGTH:
    raise ValueError(
        f"Longest formatted example is {max_observed} tokens, exceeding "
        f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. No record will be silently truncated."
    )

for name, lengths in (("train", train_lengths), ("validation", validation_lengths)):
    print(
        f"{name}: min={min(lengths)}, median={int(np.median(lengths))}, "
        f"p95={int(np.percentile(lengths, 95))}, max={max(lengths)} tokens"
    )

probe = train_text_rows[0]["text"]
assert "<|im_start|>assistant\n" in probe, "Unexpected Qwen assistant template marker"
assert "<think>\n" in probe and "</think>\n\n\\boxed{" in probe


## 5. Supervised fine-tuning

The trainer masks every token before the assistant response. Consequently, validation loss is computed on the gold boxed answer rather than on the long question text. Packing is disabled so examples and masks cannot cross sequence boundaries.


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    dataset_num_proc=1,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Verify that masking leaves assistant targets while excluding prompt tokens.
encoded_probe = tokenizer(train_text_rows[0]["text"], add_special_tokens=False)
masked_probe = trainer.data_collator([encoded_probe])["labels"][0]
trained_tokens = int((masked_probe != -100).sum().item())
assert 0 < trained_tokens < len(masked_probe), "Completion-only masking sanity check failed"
print(f"Mask check: {trained_tokens}/{len(masked_probe)} tokens contribute to loss")


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best held-out loss:", trainer.state.best_metric)


## 6. Full generated-label evaluation

This is the primary held-out metric. The model receives only the system prompt and question, generates its own reasoning and final answer, and receives credit only when the final non-whitespace text is a boxed `C1`–`C8` label.

Running all 864 long prompts on a T4 can take substantial time. Set `EVAL_LIMIT` in Section 2 only for debugging; leave it as `None` for the official result.


In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)
model.eval()

STRICT_FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}\s*$")
evaluation_records = validation_rows if EVAL_LIMIT is None else validation_rows[:EVAL_LIMIT]
predictions = []

for row in tqdm(evaluation_records, desc="Generating validation answers"):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()
    match = STRICT_FINAL_BOX_RE.search(completion)
    prediction = match.group(1) if match else None
    predictions.append(
        {
            "id": row["id"],
            "target": row["target"],
            "prediction": prediction,
            "correct": prediction == row["target"],
            "format_valid": prediction is not None,
            "completion": completion,
        }
    )

correct = sum(row["correct"] for row in predictions)
format_valid = sum(row["format_valid"] for row in predictions)
total = len(predictions)
accuracy = correct / total
format_valid_rate = format_valid / total
print(f"Exact generated-label accuracy: {accuracy:.2%} ({correct}/{total})")
print(f"Strict boxed-format rate:       {format_valid_rate:.2%} ({format_valid}/{total})")


In [ ]:
import pandas as pd

prediction_frame = pd.DataFrame(predictions)
labels = [f"C{i}" for i in range(1, 9)]
confusion = pd.crosstab(
    prediction_frame["target"],
    prediction_frame["prediction"].fillna("MALFORMED"),
    dropna=False,
).reindex(index=labels, columns=labels + ["MALFORMED"], fill_value=0)

per_class = (
    prediction_frame.groupby("target", sort=True)["correct"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "correct", "mean": "accuracy"})
)
display(per_class)
display(confusion)

metrics = {
    "model": MODEL_NAME,
    "adapter": ADAPTER_DIR,
    "questions": total,
    "full_validation_set": EVAL_LIMIT is None,
    "correct": correct,
    "accuracy": accuracy,
    "format_valid": format_valid,
    "format_valid_rate": format_valid_rate,
    "per_class": {
        label: {
            "correct": int(per_class.loc[label, "correct"]),
            "count": int(per_class.loc[label, "count"]),
            "accuracy": float(per_class.loc[label, "accuracy"]),
        }
        for label in per_class.index
    },
    "confusion_matrix": {
        target: {predicted: int(confusion.loc[target, predicted]) for predicted in confusion.columns}
        for target in confusion.index
    },
}

with open("validation_predictions.jsonl", "w", encoding="utf-8") as handle:
    for row in predictions:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
with open("validation_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, ensure_ascii=False, indent=2)
print("Saved validation_predictions.jsonl and validation_metrics.json")


## 7. Save the adapter and evaluation artifacts

The resulting directory is a LoRA adapter, not a merged model. Later GRPO must load it over `unsloth/Qwen3.5-4B` with `fast_inference=False` and preserve the native `<think>...</think>\n\n\boxed{...}` contract. Do not reuse a Qwen3-4B-Base GRPO notebook or its custom `<start_working_out>` tags unchanged.


In [ ]:
import shutil

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(Path(ADAPTER_DIR) / "training_handoff.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "base_model": MODEL_NAME,
            "max_seq_length": MAX_SEQ_LENGTH,
            "load_in_4bit": False,
            "load_in_16bit": True,
            "fast_inference": False,
            "response_contract": "<think>...</think>\\n\\n\\boxed{candidate}",
            "grpo_note": (
                "Load this adapter over the same Qwen3.5 base and tokenizer. "
                "Keep native thinking tags and reward the strict final boxed label."
            ),
        },
        handle,
        indent=2,
    )

adapter_archive = shutil.make_archive(ADAPTER_DIR, "zip", root_dir=ADAPTER_DIR)
print("Adapter archive:", adapter_archive)
print("Evaluation files: validation_predictions.jsonl, validation_metrics.json")

try:
    from google.colab import files
    files.download(adapter_archive)
    files.download("validation_metrics.json")
    files.download("validation_predictions.jsonl")
except ImportError:
    print("Not running in Colab; artifacts remain in the current directory.")
